In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.model import ECG_CNN1D

dispositivo = torch.device('cpu')  # Grad-CAM sobre un solo ejemplo no necesita GPU

CLASES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

In [ ]:
modelo = ECG_CNN1D(num_leads=12, num_classes=5).to(dispositivo)
modelo.load_state_dict(torch.load('../models/mejor_modelo.pt', map_location=dispositivo, weights_only=True))
modelo.eval()

# Queremos "espiar" justo después de la última capa ReLU del bloque3,
# antes de que el AdaptiveAvgPool1d comprima toda la información temporal.
# Recuerda la arquitectura: bloque3 = [Conv1d, BatchNorm1d, ReLU, AdaptiveAvgPool1d]
capa_objetivo = modelo.bloque3[2]

from src.gradcam import GradCAM1D
gradcam = GradCAM1D(modelo, capa_objetivo)

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import ECGDataset
from src.train import evaluar
from src.losses import FocalLossMultiLabel

X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

test_dataset = ECGDataset('../data/processed/X_test.npy', '../data/processed/y_test.npy')
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

_, predicciones_test, etiquetas_test = evaluar(modelo, test_loader, FocalLossMultiLabel(), dispositivo)

In [ ]:
indice_clase = CLASES.index('HYP')

positivos_reales = np.where(etiquetas_test[:, indice_clase] == 1)[0]
probas_de_esos_casos = predicciones_test[positivos_reales, indice_clase]
idx_ejemplo = positivos_reales[np.argmax(probas_de_esos_casos)]

print(f"Índice del ejemplo elegido: {idx_ejemplo}")
print(f"Probabilidad predicha para HYP: {predicciones_test[idx_ejemplo, indice_clase]:.4f}")

In [ ]:
def graficar_gradcam(señal, cam, derivacion=1, nombre_clase='', probabilidad=None, ruta_guardado=None):
    fig, ax = plt.subplots(figsize=(14, 4))

    ax.imshow(
        cam[np.newaxis, :], cmap='Reds', aspect='auto', alpha=0.5,
        extent=[0, len(cam), señal[:, derivacion].min(), señal[:, derivacion].max()]
    )
    ax.plot(señal[:, derivacion], color='black', linewidth=1.3)

    titulo = f'Grad-CAM — clase {nombre_clase}'
    if probabilidad is not None:
        titulo += f' (probabilidad predicha: {probabilidad:.2f})'
    ax.set_title(titulo)
    ax.set_xlabel('Muestras (tiempo)')
    ax.set_ylabel('Amplitud normalizada')

    if ruta_guardado:
        fig.savefig(ruta_guardado, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
import os
os.makedirs('../assets', exist_ok=True)

señal_ejemplo = X_test[idx_ejemplo]  # forma (1000, 12), ya preprocesada
señal_tensor = torch.tensor(señal_ejemplo, dtype=torch.float32).transpose(0, 1)  # (12, 1000)

cam, probabilidad = gradcam.generar(señal_tensor, indice_clase)

graficar_gradcam(
    señal_ejemplo, cam,
    derivacion=1, nombre_clase='HYP', probabilidad=probabilidad,
    ruta_guardado='../assets/gradcam_hyp_ejemplo.png'
)

In [ ]:
for nombre_clase in ['NORM', 'MI']:
    indice_clase = CLASES.index(nombre_clase)
    positivos_reales = np.where(etiquetas_test[:, indice_clase] == 1)[0]
    probas_de_esos_casos = predicciones_test[positivos_reales, indice_clase]
    idx_ejemplo = positivos_reales[np.argmax(probas_de_esos_casos)]

    señal_ejemplo = X_test[idx_ejemplo]
    señal_tensor = torch.tensor(señal_ejemplo, dtype=torch.float32).transpose(0, 1)
    cam, probabilidad = gradcam.generar(señal_tensor, indice_clase)

    graficar_gradcam(
        señal_ejemplo, cam,
        derivacion=1, nombre_clase=nombre_clase, probabilidad=probabilidad,
        ruta_guardado=f'../assets/gradcam_{nombre_clase.lower()}_ejemplo.png'
    )